# MCD-rPPG Dataset Preprocessing Notebook

**Database-Driven Preprocessing for MCD-rPPG Dataset**

This notebook processes the MCD-rPPG dataset using the database CSV file (db.csv) which contains all file paths and metadata.

## Dataset Structure

Based on the database CSV, the dataset has this structure:

### Database Columns:
- **patient_id**: Subject identifier (e.g., 1020)
- **step**: Condition - 'before' (resting) or 'after' (post-exercise)
- **camera**: Camera name - 'IriunWebcam', 'FullHDwebcam', 'USBVideo'
- **view**: View type - 'front', 'left', 'right'
- **video**: Relative path to video file (e.g., 'video/1020_FullHDwebcam_after.avi')
- **ppg**: Relative path to PPG file (e.g., 'ppg/1020_after.PW')
- **meta**: Relative path to meta file (e.g., 'meta/1020_FullHDwebcam_after.txt')
- **ppg_sync**: Relative path to PPG sync file (e.g., 'ppg_sync/1020_FullHDwebcam_after.txt')
- **ecg**: Relative path to ECG file (e.g., 'ecg/1020_after.json')
- **Vitals**: weight, height, bmi, age, sex, upper_ap, lower_ap, saturation, temperature, pulse, stress

### File Formats:
- **Video**: .avi files with ~30 FPS
- **PPG**: .PW files with format: `{value} {timestamp}`
- **Meta**: .txt files with format: `{frame_number} {timestamp}`
- **PPG Sync**: .txt files with format: `{frame_number} {sync_value}`
- **ECG**: .json files with ECG data

### Key Observations:
- PPG files are **per patient+step** (shared across all cameras for same patient and step)
- Video, Meta, PPG Sync files are **per patient+camera+step**
- ECG files are **per patient+step** (same as PPG)

## Preprocessing Pipeline

1. Load database CSV (db.csv)
2. For each row, load matching files
3. Parse PPG from .PW file with timestamps
4. Parse meta file for video frame timestamps
5. Align PPG with video using timestamp interpolation
6. Process video frames with MediaPipe face detection
7. Extract chunks for training
8. Save preprocessed data with metadata


---

## 1. Setup and Configuration

### Install Dependencies

In [ ]:
# Install required packages
!pip install mediapipe>=0.10.11 opencv-python-headless tqdm scikit-image numpy pandas scipy matplotlib

### Import Libraries

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from datetime import datetime
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from sklearn.model_selection import train_test_split

# MediaPipe imports
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Visualization
import matplotlib.pyplot as plt

print('All imports successful!')
print(f'MediaPipe version: {mp.__version__}')

### Configuration

In [ ]:
class Config:
    # Paths
    db_path = None  # Path to db.csv
    dataset_root = None  # Root path to dataset
    output_path = './preprocessed_data'
    
    # Processing
    window_size = 256
    stride = 64
    target_size = (128, 128)
    padding = 20
    min_face_size = 64
    
    # Signal processing
    ppg_low_freq = 0.75
    ppg_high_freq = 4.0
    frame_rate = 30.0
    ppg_rate = 100.0
    
    # Splits
    train_ratio = 0.8
    val_ratio = 0.1
    test_ratio = 0.1
    random_state = 42

config = Config()
print('Configuration initialized')

### Set Paths

**Set these to your actual paths:**

In [ ]:
# Set paths to your dataset
config.db_path = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
config.dataset_root = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88'

# Verify paths
if os.path.exists(config.db_path):
    print(f'Database CSV found: {config.db_path}')
else:
    print('ERROR: Database CSV not found!')

if os.path.exists(config.dataset_root):
    print(f'Dataset root found: {config.dataset_root}')
else:
    print('ERROR: Dataset root not found!')

---

## 2. Load Database

Load the database CSV which contains all file paths and metadata.

In [ ]:
# Load database
db = pd.read_csv(config.db_path)
print(f'Loaded {len(db)} rows from database')
print(f'Columns: {list(db.columns)}')

# Show first few rows
display(db.head())

---

## 3. Initialize MediaPipe

Initialize the MediaPipe Face Landmarker for face detection.

In [ ]:
def initialize_mediapipe():
    base_options = python.BaseOptions(model_asset_path=None)
    options = vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_faces=1
    )
    detector = vision.FaceLandmarker.create_from_options(options)
    return detector

detector = initialize_mediapipe()
print('MediaPipe Face Landmarker initialized')

---

## 4. File Parsing Functions

Functions to parse the various file formats in the dataset.

In [ ]:
def parse_pw_file(pw_path):
    """Parse .PW file: {ppg_value}   {timestamp}"""
    ppg_values, timestamps = [], []
    with open(pw_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    ppg_values.append(float(parts[0]))
                    timestamp = datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f')
                    timestamps.append(timestamp)
                except:
                    pass
    return np.array(ppg_values), np.array(timestamps)

In [ ]:
def parse_meta_file(meta_path):
    """Parse meta file: {frame_number}  {timestamp}"""
    meta_data = {}
    with open(meta_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    frame_num = int(parts[0])
                    timestamp = datetime.strptime(' '.join(parts[1:]), '%Y-%m-%d %H:%M:%S.%f')
                    meta_data[frame_num] = timestamp
                except:
                    pass
    return meta_data

In [ ]:
def parse_ppg_sync_file(sync_path):
    """Parse PPG sync file: {frame_number} {sync_value}"""
    sync_data = {}
    with open(sync_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                try:
                    frame_num = int(parts[0])
                    sync_value = float(parts[1])
                    sync_data[frame_num] = sync_value
                except:
                    pass
    return sync_data

---

## 5. PPG-Video Alignment

The key function that aligns PPG signals with video frames using timestamps.

In [ ]:
def align_ppg_with_video(ppg_values, ppg_timestamps, meta_data, frame_rate=30.0):
    """
    Align PPG signal with video frames using timestamp interpolation.
    
    This is the CORE alignment function that:
    1. Converts all timestamps to seconds relative to start
    2. Uses linear interpolation to estimate PPG values at video frame timestamps
    3. Returns PPG signal aligned with video frames
    
    Args:
        ppg_values: Array of PPG values from .PW file
        ppg_timestamps: Array of timestamps for each PPG value
        meta_data: Dict mapping frame_number to timestamp
        frame_rate: Video frame rate (FPS)
    
    Returns:
        aligned_ppg: Array of PPG values aligned with video frames
    """
    if len(meta_data) == 0 or len(ppg_timestamps) == 0:
        return np.zeros(len(meta_data))
    
    # Convert all timestamps to seconds relative to first timestamp
    first_ppg = ppg_timestamps[0]
    first_meta = list(meta_data.values())[0]
    
    # PPG timestamps in seconds
    ppg_seconds = [(ts - first_ppg).total_seconds() for ts in ppg_timestamps]
    
    # Sort meta data by frame number and convert to seconds
    sorted_frames = sorted(meta_data.keys())
    meta_seconds = [(meta_data[f] - first_meta).total_seconds() for f in sorted_frames]
    
    # Interpolate PPG values at video frame timestamps
    interp_func = interp1d(ppg_seconds, ppg_values, kind='linear', fill_value='extrapolate')
    aligned_ppg = interp_func(meta_seconds)
    
    return aligned_ppg

In [ ]:
def preprocess_ppg(ppg, low_freq=0.75, high_freq=4.0, fs=100.0):
    """Apply bandpass filter and normalize PPG signal."""
    nyquist = 0.5 * fs
    low = low_freq / nyquist
    high = high_freq / nyquist
    b, a = butter(4, [low, high], btype='band')
    filtered = filtfilt(b, a, ppg)
    return (filtered - filtered.mean()) / filtered.std()

---

## 6. Video Processing Functions

Functions for loading and processing video frames.

In [ ]:
def load_video(video_path):
    """Load video file and return frames as numpy array."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise ValueError(f'Could not open: {video_path}')
    
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
    
    cap.release()
    return np.array(frames, dtype=np.uint8)

In [ ]:
def detect_face(frame, detector, frame_idx):
    """Detect face landmarks using MediaPipe."""
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
    result = detector.detect_for_video(mp_image, frame_idx)
    if result and result.face_landmarks:
        lm = result.face_landmarks[0]
        h, w = frame.shape[:2]
        return np.array([(p.x * w, p.y * h) for p in lm], dtype=np.float32)
    return None

def process_frame(frame, detector, frame_idx, prev_landmarks=None, min_size=64):
    """Process single frame: detect face, crop, resize."""
    landmarks = detect_face(frame, detector, frame_idx)
    if landmarks is None:
        return None, prev_landmarks if prev_landmarks is not None else None
    
    x_min, x_max = landmarks[:, 0].min(), landmarks[:, 0].max()
    y_min, y_max = landmarks[:, 1].min(), landmarks[:, 1].max()
    w, h = x_max - x_min, y_max - y_min
    
    if w < min_size or h < min_size:
        return None, landmarks
    
    pad = 20
    x_min = max(0, int(x_min) - pad)
    x_max = min(frame.shape[1], int(x_max) + pad)
    y_min = max(0, int(y_min) - pad)
    y_max = min(frame.shape[0], int(y_max) + pad)
    
    face = frame[y_min:y_max, x_min:x_max]
    face = cv2.resize(face, (128, 128))
    return face, landmarks

In [ ]:
def extract_chunks(video, ppg, window_size, stride, video_fps=30.0, ppg_fps=100.0):
    """Extract chunks from video and PPG for training."""
    chunks, ppg_chunks = [], []
    num_frames = video.shape[0]
    ppg_per_frame = ppg_fps / video_fps
    
    for start in range(0, num_frames - window_size + 1, stride):
        end = start + window_size
        video_chunk = video[start:end]
        ppg_start = int(start * ppg_per_frame)
        ppg_end = int(end * ppg_per_frame)
        ppg_chunk = ppg[ppg_start:ppg_end]
        if len(ppg_chunk) == window_size:
            chunks.append(video_chunk)
            ppg_chunks.append(ppg_chunk)
    
    return np.array(chunks), np.array(ppg_chunks)

---

## 7. Process Dataset

Process all rows from the database.

In [ ]:
# Create output directory
os.makedirs(config.output_path, exist_ok=True)

# Initialize counters
all_video_chunks, all_ppg_chunks, all_metadata = [], [], []
processed_count = 0
failed_count = 0

# Process each row from database
for idx, row in tqdm(db.iterrows(), total=len(db), desc='Processing'):
    try:
        # Get file paths from database
        video_path = os.path.join(config.dataset_root, row['video'])
        ppg_path = os.path.join(config.dataset_root, row['ppg'])
        meta_path = os.path.join(config.dataset_root, row['meta'])
        ppg_sync_path = os.path.join(config.dataset_root, row['ppg_sync'])
        
        # Check files exist
        if not all(os.path.exists(p) for p in [video_path, ppg_path, meta_path]):
            print(f'  Missing files for patient {row["patient_id"]}')
            failed_count += 1
            continue
        
        # Load data
        video = load_video(video_path)
        ppg_values, ppg_timestamps = parse_pw_file(ppg_path)
        meta_data = parse_meta_file(meta_path)
        
        # Align PPG with video
        aligned_ppg = align_ppg_with_video(ppg_values, ppg_timestamps, meta_data, config.frame_rate)
        processed_ppg = preprocess_ppg(aligned_ppg, config.ppg_low_freq, config.ppg_high_freq, config.ppg_rate)
        
        # Process video frames
        processed_frames, all_lms = [], []
        prev_lms = None
        
        for fi, frame in enumerate(video):
            pf, lms = process_frame(frame, detector, fi, prev_lms, config.min_face_size)
            if pf is not None:
                processed_frames.append(pf)
                all_lms.append(lms)
                prev_lms = lms
        
        if len(processed_frames) < config.window_size:
            print(f'  Too few frames for patient {row["patient_id"]}')
            failed_count += 1
            continue
        
        # Extract chunks
        video_chunks, ppg_chunks = extract_chunks(
            np.array(processed_frames), processed_ppg,
            config.window_size, config.stride,
            config.frame_rate, config.ppg_rate
        )
        
        all_video_chunks.extend(video_chunks)
        all_ppg_chunks.extend(ppg_chunks)
        
        # Save metadata for each chunk
        for ci in range(len(video_chunks)):
            all_metadata.append({
                'patient_id': row['patient_id'],
                'step': row['step'],
                'camera': row['camera'],
                'view': row['view'],
                'chunk_idx': ci,
                'num_frames': len(video),
                'ppg_length': len(aligned_ppg),
                'weight': row['weight'],
                'height': row['height'],
                'bmi': row['bmi'],
                'age': row['age'],
                'sex': row['sex'],
                'upper_ap': row['upper_ap'],
                'lower_ap': row['lower_ap'],
                'saturation': row['saturation'],
                'temperature': row['temperature'],
                'pulse': row['pulse'],
                'stress': row['stress']
            })
        
        processed_count += 1
        
    except Exception as e:
        print(f'  Error processing patient {row["patient_id"]}: {str(e)[:100]}')
        failed_count += 1

print(f'Processed: {processed_count}/{len(db)}, Failed: {failed_count}')

---

## 8. Save Preprocessed Data

Save all preprocessed data to disk.

In [ ]:
# Save video chunks
if all_video_chunks:
    np.save(os.path.join(config.output_path, 'video_chunks.npy'), np.array(all_video_chunks))
    print(f'Saved {len(all_video_chunks)} video chunks with shape {np.array(all_video_chunks).shape}')

# Save PPG chunks
if all_ppg_chunks:
    np.save(os.path.join(config.output_path, 'ppg_chunks.npy'), np.array(all_ppg_chunks))
    print(f'Saved {len(all_ppg_chunks)} PPG chunks with shape {np.array(all_ppg_chunks).shape}')

# Save metadata
if all_metadata:
    metadata_df = pd.DataFrame(all_metadata)
    metadata_df.to_csv(os.path.join(config.output_path, 'metadata.csv'), index=False)
    print(f'Saved metadata for {len(all_metadata)} chunks')

# Create train/val/test splits by patient_id
patient_ids = metadata_df['patient_id'].unique()
train_patients, test_patients = train_test_split(
    patient_ids, test_size=config.test_ratio + config.val_ratio, random_state=config.random_state
)
val_patients, test_patients = train_test_split(
    test_patients, test_size=config.test_ratio / (config.test_ratio + config.val_ratio), random_state=config.random_state
)

for split_name, patients in [('train', train_patients), ('val', val_patients), ('test', test_patients)]:
    with open(os.path.join(config.output_path, f'{split_name}_patients.txt'), 'w') as f:
        for p in patients:
            f.write(f'{p}\n')
    print(f'Saved {split_name} split: {len(patients)} patients')

print('All data saved successfully!')

---

## 9. Verify Output

Check that all output files were created correctly.

In [ ]:
# Check output files
output_files = [
    'video_chunks.npy',
    'ppg_chunks.npy',
    'metadata.csv',
    'train_patients.txt',
    'val_patients.txt',
    'test_patients.txt'
]

for fname in output_files:
    fpath = os.path.join(config.output_path, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / (1024 * 1024)
        print(f'  {fname}: {size:.2f} MB')
    else:
        print(f'  {fname}: NOT FOUND')

# Load and check a sample
if os.path.exists(os.path.join(config.output_path, 'video_chunks.npy')):
    video_chunks = np.load(os.path.join(config.output_path, 'video_chunks.npy'))
    ppg_chunks = np.load(os.path.join(config.output_path, 'ppg_chunks.npy'))
    metadata = pd.read_csv(os.path.join(config.output_path, 'metadata.csv'))
    
    print(f'\nSample video chunk shape: {video_chunks[0].shape}')
    print(f'Sample PPG chunk shape: {ppg_chunks[0].shape}')
    print(f'\nFirst metadata row:')
    display(metadata.head(1))

In [ ]:
# Visualize a sample
if len(all_video_chunks) > 0:
    plt.figure(figsize=(15, 5))
    for i in range(min(5, all_video_chunks[0].shape[0])):
        plt.subplot(1, 5, i+1)
        plt.imshow(all_video_chunks[0][i])
        plt.title(f'Frame {i}')
        plt.axis('off')
    plt.suptitle('Sample Video Chunk')
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(12, 4))
    plt.plot(all_ppg_chunks[0], label='PPG Signal')
    plt.title('Sample PPG Chunk')
    plt.xlabel('Sample')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True)
    plt.show()

---

## Summary

### What This Notebook Does:

1. **Loads database CSV** (db.csv) which contains all file paths
2. **Parses .PW files** to extract PPG values and timestamps
3. **Parses meta files** to get video frame timestamps
4. **Aligns PPG with video** using timestamp interpolation
5. **Processes video frames** with MediaPipe face detection
6. **Extracts chunks** for training
7. **Saves preprocessed data** with full metadata including vitals
8. **Creates train/val/test splits** by patient_id

### Output Files:

```
preprocessed_data/
├── video_chunks.npy      # Shape: (N, window_size, 128, 128, 3)
├── ppg_chunks.npy        # Shape: (N, window_size)
├── metadata.csv         # Full metadata including vitals
├── train_patients.txt   # Training split patient IDs
├── val_patients.txt     # Validation split patient IDs
└── test_patients.txt    # Test split patient IDs
```

### Key Features:

- **Database-driven**: Uses db.csv to find all files
- **Timestamp alignment**: Aligns PPG with video using precise timestamps
- **Vitals included**: Saves all biomarker data in metadata
- **MediaPipe**: Uses 468-point face mesh for detection
- **Proper splits**: Train/val/test by patient_id to avoid data leakage

---

**Last Updated**: September 2025
**Dataset**: MCD-rPPG from Bgeorge/mcd_rppg
**License**: MIT